In [8]:
import os

IGNORAR = {"__pycache__", "data"}  # adicione aqui o que quiser ignorar

for root, dirs, files in os.walk(".."):
    dirs[:] = [d for d in dirs if not d.startswith(".") and d not in IGNORAR]
    level = root.replace(".", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        size_str = f"{size/1024/1024:.1f}MB" if size > 1024*1024 else f"{size/1024:.0f}KB"
        print(f"{indent}  {f}  ({size_str})")

../
  .gitignore  (0KB)
  README.md  (4KB)
  requirements.txt  (0KB)
  fase2/
    track2_download_link_1/
      Guide to the Second Round_track2/
        Round2 Project Submission Manual_cn_0407.pdf  (2.5MB)
        Round2 Project Submission Manual_en_0407.pdf  (2.2MB)
        Space Intelligence for Clean Water Track_Task Description of Final Round_cn.pdf  (1.2MB)
        Space Intelligence for Clean Water Track_Task Description of Final Round_en.pdf  (1.1MB)
        test_input_sample/
          track2_cha_test_point.csv  (0KB)
          track2_turb_test_point.csv  (0KB)
          area8_images/
            area8_2024-01-15.tif  (60.4MB)
    track2_download_link_2/
      area6/
        track2_cha_train_point_area6.csv  (1KB)
        track2_turb_train_point_area6.csv  (1KB)
        area6_images/
          area6_2024-01-15.tif  (53.6MB)
          area6_2024-02-04.tif  (209.0MB)
          area6_2024-02-09.tif  (235KB)
          area6_2024-02-14.tif  (177KB)
          area6_2024-02-19.tif  

In [9]:
import rasterio
import pandas as pd
import numpy as np

# testando inspecionar tifs
for path in [
    "../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-04.tif",
    "../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-09.tif",
]:
    with rasterio.open(path) as src:
        print(f"\n{path.split('/')[-1]}")
        print(f"  bandas : {src.count}")
        print(f"  shape  : {src.shape}  (height x width)")
        print(f"  CRS    : {src.crs}")
        print(f"  res    : {src.res}")
        print(f"  dtype  : {src.dtypes[0]}")
        print(f"  bounds : {src.bounds}")
        # lê banda 1 e mostra range de valores
        b1 = src.read(1).astype(float)
        if src.nodata is not None:
            b1[b1 == src.nodata] = np.nan
        print(f"  min/max banda1: {np.nanmin(b1):.1f} / {np.nanmax(b1):.1f}")

# inspeciona csv de treino
for path in [
    "../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv",
    "../fase2/track2_download_link_2/area6/track2_cha_train_point_area6.csv",
    "../fase2/track2_download_link_2/area7/track2_turb_train_point_area7.csv",
]:
    df = pd.read_csv(path)
    print(f"\n{path.split('/')[-1]}")
    print(df.head(3))
    print(f"  shape: {df.shape}")
    col = [c for c in df.columns if c not in ('filename','Lon','Lat')][0]
    print(f"  {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")


area6_2024-02-04.tif
  bandas : 12
  shape  : (2500, 2130)  (height x width)
  CRS    : EPSG:4326
  res    : (0.00032559248826291, 0.00024343839999999944)
  dtype  : float32
  bounds : BoundingBox(left=-75.343552, bottom=41.280903, right=-74.65004, top=41.889499)
  min/max banda1: 0.0 / 1.2

area6_2024-02-09.tif
  bandas : 12
  shape  : (81, 76)  (height x width)
  CRS    : EPSG:4326
  res    : (9.009210526325786e-05, 6.664197530863667e-05)
  dtype  : float32
  bounds : BoundingBox(left=-75.217938, bottom=41.865009, right=-75.211091, top=41.870407)
  min/max banda1: 0.0 / 0.5

track2_turb_train_point_area6.csv
               filename        Lon        Lat  turb_value
0  area6_2024-01-15.tif -74.795580  41.309385         8.6
1  area6_2024-02-04.tif -74.795417  41.309264         3.1
2  area6_2024-02-14.tif -74.795108  41.309196         1.8
  shape: (31, 4)
  turb_value: min=0.50, max=8.60, mean=1.99

track2_cha_train_point_area6.csv
               filename        Lon        Lat  cha_val

In [10]:
import rasterio
import numpy as np
import pandas as pd

# 1. confirmar o que são as 12 bandas (nomes ou descrições se existirem)
with rasterio.open("../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-04.tif") as src:
    print("Descrições das bandas:")
    for i, desc in enumerate(src.descriptions, 1):
        print(f"  banda {i}: {desc}")
    print("\nTags:", src.tags())

    # 2. pegar um ponto do CSV e ver se o pixel correspondente tem valores válidos
    df = pd.read_csv("../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv")
    row = df[df['filename'] == 'area6_2024-02-04.tif'].iloc[0]
    lon, lat = row['Lon'], row['Lat']

    py, px = src.index(lon, lat)
    print(f"\nPonto: lon={lon}, lat={lat}")
    print(f"Pixel correspondente: row={py}, col={px}")
    print(f"Shape da imagem: {src.shape}")

    # extrai patch 11x11 centrado no ponto
    PATCH = 11
    half = PATCH // 2
    window = rasterio.windows.Window(px - half, py - half, PATCH, PATCH)
    patch = src.read(window=window)  # shape: (12, 11, 11)
    print(f"\nShape do patch: {patch.shape}")
    print(f"Valores banda 1 (pixel central): {patch[0, half, half]:.4f}")
    print(f"Valores todas as bandas no pixel central:")
    print(np.round(patch[:, half, half], 4))

Descrições das bandas:
  banda 1: None
  banda 2: None
  banda 3: None
  banda 4: None
  banda 5: None
  banda 6: None
  banda 7: None
  banda 8: None
  banda 9: None
  banda 10: None
  banda 11: None
  banda 12: None

Tags: {'AREA_OR_POINT': 'Area'}

Ponto: lon=-74.795417, lat=41.309264
Pixel correspondente: row=2383, col=1683
Shape da imagem: (2500, 2130)

Shape do patch: (12, 11, 11)
Valores banda 1 (pixel central): 0.0072
Valores todas as bandas no pixel central:
[0.0072 0.0109 0.0164 0.0157 0.0159 0.0146 0.0156 0.0181 0.0202 0.0823
 0.0452 0.0293]


In [11]:
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# mapeamento de todos os pares (imagem, csv_turb, csv_cha)
AREAS = {
    'area1': {
        'img_dir': '../fase2/track2_download_link_5/area1/area1_images',
        'turb':    '../fase2/track2_download_link_5/area1/track2_turb_train_point_area1.csv',
        'cha':     '../fase2/track2_download_link_5/area1/track2_cha_train_point_area1.csv',
    },
    'area2': {
        'img_dir': '../fase2/track2_download_link_4/area2/area2_images',
        'turb':    '../fase2/track2_download_link_4/area2/track2_turb_train_point_area2.csv',
        'cha':     None,
    },
    'area3': {
        'img_dir': '../fase2/track2_download_link_3/area3/area3_images',
        'turb':    '../fase2/track2_download_link_3/area3/track2_turb_train_point_area3.csv',
        'cha':     None,
    },
    'area5': {
        'img_dir': '../fase2/track2_download_link_3/area5/area5_images',
        'turb':    '../fase2/track2_download_link_3/area5/track2_turb_train_point_area5.csv',
        'cha':     '../fase2/track2_download_link_3/area5/track2_cha_train_point_area5.csv',
    },
    'area6': {
        'img_dir': '../fase2/track2_download_link_2/area6/area6_images',
        'turb':    '../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv',
        'cha':     '../fase2/track2_download_link_2/area6/track2_cha_train_point_area6.csv',
    },
    'area7': {
        'img_dir': '../fase2/track2_download_link_2/area7/area7_images',
        'turb':    '../fase2/track2_download_link_2/area7/track2_turb_train_point_area7.csv',
        'cha':     '../fase2/track2_download_link_2/area7/track2_cha_train_point_area7.csv',
    },
}

# 1. contagem total de amostras por alvo
print("=" * 50)
print("AMOSTRAS POR ÁREA E ALVO")
print("=" * 50)
total_turb, total_cha = 0, 0
for area, cfg in AREAS.items():
    n_turb, n_cha = 0, 0
    if cfg['turb']:
        n_turb = len(pd.read_csv(cfg['turb']))
        total_turb += n_turb
    if cfg['cha']:
        n_cha = len(pd.read_csv(cfg['cha']))
        total_cha += n_cha
    print(f"  {area}: turb={n_turb:>4}  cha={n_cha:>4}")
print(f"  TOTAL : turb={total_turb:>4}  cha={total_cha:>4}")

# 2. distribuição dos targets (com e sem log)
print("\n" + "=" * 50)
print("DISTRIBUIÇÃO DOS TARGETS")
print("=" * 50)
for target in ['turb', 'cha']:
    vals = []
    for area, cfg in AREAS.items():
        if cfg[target]:
            df = pd.read_csv(cfg[target])
            col = 'turb_value' if target == 'turb' else 'cha_value'
            vals.extend(df[col].dropna().tolist())
    vals = np.array(vals)
    print(f"\n  {target} (escala original):")
    print(f"    n={len(vals)}  min={vals.min():.2f}  max={vals.max():.2f}"
          f"  mean={vals.mean():.2f}  median={np.median(vals):.2f}  std={vals.std():.2f}")
    log_vals = np.log1p(vals)
    print(f"  {target} (log1p):")
    print(f"    min={log_vals.min():.3f}  max={log_vals.max():.3f}"
          f"  mean={log_vals.mean():.3f}  std={log_vals.std():.3f}")

# 3. verificar se todos os TIFs referenciados nos CSVs existem
print("\n" + "=" * 50)
print("ARQUIVOS FALTANDO")
print("=" * 50)
missing = 0
for area, cfg in AREAS.items():
    for target in ['turb', 'cha']:
        if not cfg[target]:
            continue
        df = pd.read_csv(cfg[target])
        for fname in df['filename'].unique():
            fpath = Path(cfg['img_dir']) / fname
            if not fpath.exists():
                print(f"  FALTA: {area}/{fname}")
                missing += 1
if missing == 0:
    print("  nenhum arquivo faltando")

# 4. verificar TIFs minúsculos (provavelmente inutilizáveis)
print("\n" + "=" * 50)
print("TIFs MUITO PEQUENOS (shape < 100x100)")
print("=" * 50)
for area, cfg in AREAS.items():
    for tif in sorted(Path(cfg['img_dir']).glob('*.tif')):
        with rasterio.open(tif) as src:
            h, w = src.shape
            if h < 100 or w < 100:
                print(f"  {area}/{tif.name}: {h}x{w}")

AMOSTRAS POR ÁREA E ALVO
  area1: turb= 211  cha=  91
  area2: turb= 200  cha=   0
  area3: turb= 247  cha=   0
  area5: turb=  98  cha=  28
  area6: turb=  31  cha=  12
  area7: turb= 204  cha= 174
  TOTAL : turb= 991  cha= 305

DISTRIBUIÇÃO DOS TARGETS

  turb (escala original):
    n=991  min=0.50  max=4000.00  mean=39.27  median=15.30  std=175.21
  turb (log1p):
    min=0.405  max=8.294  mean=2.808  std=1.138

  cha (escala original):
    n=305  min=0.01  max=75.90  mean=11.99  median=8.00  std=12.05
  cha (log1p):
    min=0.010  max=4.343  mean=2.134  std=0.986

ARQUIVOS FALTANDO
  nenhum arquivo faltando

TIFs MUITO PEQUENOS (shape < 100x100)
  area5/area5_2024-02-29.tif: 38x46
  area5/area5_2024-07-13.tif: 38x46
  area6/area6_2024-02-09.tif: 81x76
  area6/area6_2024-07-03.tif: 81x76
  area6/area6_2024-09-01.tif: 81x76
